In [7]:
import os
import struct
import math
import numpy as np
import cv2          
from tqdm import tqdm

# ============================================================
# KONFIGURASI
# ============================================================
PATH_INPUT  = "D:/semester 4/PCD/praktikum/projek/test/Assets/"
PATH_OUTPUT = "D:/semester 4/PCD/praktikum/projek/test/Assets_Prepro3/"
KATEGORI    = ["Normal", "kidneyStone"]


# ============================================================
# FUNGSI MANUAL PENGGANTI NumPy
# ============================================================

def manual_floor(arr):
    """Mengganti np.floor: bulatkan ke bawah setiap elemen."""
    if isinstance(arr, np.ndarray):
        return np.array([math.floor(x) for x in arr.flat]).reshape(arr.shape)
    else:
        return math.floor(arr)

def manual_clip(arr, a_min, a_max):
    """Mengganti np.clip."""
    if isinstance(arr, np.ndarray):
        flat = arr.flat
        return np.array([max(a_min, min(a_max, x)) for x in flat]).reshape(arr.shape)
    else:
        return max(a_min, min(a_max, arr))

def manual_minimum(a, b):
    """Mengganti np.minimum (element-wise min)."""
    flat_a = a.flat
    flat_b = b.flat
    return np.array([min(x, y) for x, y in zip(flat_a, flat_b)]).reshape(a.shape)

def manual_maximum(a, b):
    """Mengganti np.maximum (element-wise max)."""
    flat_a = a.flat
    flat_b = b.flat
    return np.array([max(x, y) for x, y in zip(flat_a, flat_b)]).reshape(a.shape)

def manual_bincount(x, minlength=0):
    """Mengganti np.bincount: hitung frekuensi nilai integer."""
    if minlength == 0:
        minlength = int(max(x)) + 1 if x.size > 0 else 0
    hist = [0] * minlength
    for val in x.flat:
        hist[int(val)] += 1
    return np.array(hist, dtype=np.float64)

def manual_cumsum(arr):
    """Mengganti np.cumsum (kumulatif)."""
    res = np.zeros_like(arr)
    s = 0.0
    for i in range(arr.size):
        s += arr.flat[i]
        res.flat[i] = s
    return res

def manual_arange(start, stop=None, step=1):
    """Mengganti np.arange."""
    if stop is None:
        stop = start
        start = 0
    length = int(math.ceil((stop - start) / step))
    return np.array([start + i*step for i in range(length)], dtype=np.float64)

def manual_meshgrid(x, y, indexing='ij'):
    """Mengganti np.meshgrid untuk kasus 2D (indexing='ij')."""
    # x dan y adalah array 1D
    ny, nx = len(y), len(x)
    X = np.zeros((ny, nx), dtype=np.int64)
    Y = np.zeros((ny, nx), dtype=np.int64)
    for i in range(ny):
        for j in range(nx):
            X[i, j] = x[j]
            Y[i, j] = y[i]
    return X, Y

# ============================================================
# FUNGSI PAD MANUAL (refleksi)
# ============================================================
def manual_pad_reflect(img, pad):
    """Padding refleksi mirip np.pad(mode='reflect')."""
    h, w = img.shape
    padded = np.zeros((h + 2*pad, w + 2*pad), dtype=img.dtype)
    # isi bagian tengah
    padded[pad:pad+h, pad:pad+w] = img
    # pad vertikal (atas & bawah)
    for i in range(pad):
        # atas: cermin dari baris pad
        padded[i, pad:pad+w] = img[pad - i - 1, :]
        # bawah: cermin dari baris h - pad + i
        padded[h + pad + i, pad:pad+w] = img[h - i - 1, :]
    # pad horizontal (kiri & kanan) untuk semua baris
    for i in range(pad):
        # kiri: cermin dari kolom pad
        padded[:, i] = padded[:, 2*pad - i - 1]
        # kanan: cermin dari kolom w+pad-1
        padded[:, w + pad + i] = padded[:, w + pad - i - 1]
    return padded


# ============================================================
# FUNGSI 1: RESIZE MANUAL (tanpa np.clip, np.floor, np.ix_, dll)
# ============================================================
def resize_manual(img, ukuran_baru):
    tinggi_lama, lebar_lama = img.shape
    lebar_baru, tinggi_baru = ukuran_baru
    img_f = img.astype(np.float64)

    skala_x = lebar_lama / lebar_baru
    skala_y = tinggi_lama / tinggi_baru

    # buat array koordinat asal (menggunakan manual_arange dan manual_clip)
    y_asal = manual_clip(
        (manual_arange(tinggi_baru) + 0.5) * skala_y - 0.5,
        0, tinggi_lama - 1
    )
    x_asal = manual_clip(
        (manual_arange(lebar_baru) + 0.5) * skala_x - 0.5,
        0, lebar_lama - 1
    )

    hasil = np.zeros((tinggi_baru, lebar_baru), dtype=np.float64)

    # loop untuk setiap piksel output
    for i in range(tinggi_baru):
        y = y_asal[i]
        y0 = int(math.floor(y))
        y1 = min(y0 + 1, tinggi_lama - 1)
        wy = y - y0

        for j in range(lebar_baru):
            x = x_asal[j]
            x0 = int(math.floor(x))
            x1 = min(x0 + 1, lebar_lama - 1)
            wx = x - x0

            # ambil 4 tetangga
            p00 = img_f[y0, x0]
            p01 = img_f[y0, x1]
            p10 = img_f[y1, x0]
            p11 = img_f[y1, x1]

            atas  = p00 * (1 - wx) + p01 * wx
            bawah = p10 * (1 - wx) + p11 * wx
            hasil[i, j] = atas * (1 - wy) + bawah * wy

    # clip hasil ke [0,255]
    return manual_clip(hasil, 0, 255).astype(np.uint8)


# ============================================================
# FUNGSI 2: CLAHE MANUAL (tanpa np.bincount, np.cumsum, np.meshgrid, dll)
# ============================================================
def clahe_manual(img, clip_limit=2.0, grid_size=(8, 8)):
    tinggi, lebar = img.shape
    gx, gy = grid_size
    tinggi_tile = tinggi // gy
    lebar_tile  = lebar // gx

    # --- buat peta tile ---
    peta_tile = np.zeros((gy, gx, 256), dtype=np.float64)

    for ty in range(gy):
        y0 = ty * tinggi_tile
        y1 = tinggi if ty == gy - 1 else y0 + tinggi_tile
        for tx in range(gx):
            x0 = tx * lebar_tile
            x1 = lebar if tx == gx - 1 else x0 + lebar_tile

            tile = img[y0:y1, x0:x1]
            jumlah_piksel = tile.size

            # histogram manual
            hist = manual_bincount(tile.ravel(), minlength=256)
            # clip limit
            batas = max(1.0, clip_limit * jumlah_piksel / 256.0)
            kelebihan = 0.0
            for i in range(256):
                if hist[i] > batas:
                    kelebihan += hist[i] - batas
                    hist[i] = batas
            # tambahkan kelebihan secara merata
            hist += kelebihan / 256.0

            # CDF manual
            cdf = manual_cumsum(hist)
            peta_tile[ty, tx] = (cdf / cdf[-1]) * 255.0

    # --- interpolasi bilinear antar tile ---
    py = manual_clip(
        manual_arange(tinggi) / tinggi_tile - 0.5,
        0, gy - 1
    )
    px = manual_clip(
        manual_arange(lebar) / lebar_tile - 0.5,
        0, gx - 1
    )

    # buat indeks tile dan bobot secara manual (tanpa meshgrid)
    hasil = np.zeros((tinggi, lebar), dtype=np.float64)

    for i in range(tinggi):
        y = py[i]
        ty0 = int(math.floor(y))
        ty1 = min(ty0 + 1, gy - 1)
        wy = y - ty0

        for j in range(lebar):
            x = px[j]
            tx0 = int(math.floor(x))
            tx1 = min(tx0 + 1, gx - 1)
            wx = x - tx0

            val = img[i, j]
            v00 = peta_tile[ty0, tx0, val]
            v01 = peta_tile[ty0, tx1, val]
            v10 = peta_tile[ty1, tx0, val]
            v11 = peta_tile[ty1, tx1, val]

            atas  = v00 * (1 - wx) + v01 * wx
            bawah = v10 * (1 - wx) + v11 * wx
            hasil[i, j] = atas * (1 - wy) + bawah * wy

    return manual_clip(hasil, 0, 255).astype(np.uint8)


# ============================================================
# FUNGSI 3: SHARPENING MANUAL (kernel 3x3, tanpa np.pad)
# ============================================================
def sharpening_manual(img):
    kernel = np.array([[ 0, -1,  0],
                       [-1,  5, -1],
                       [ 0, -1,  0]], dtype=np.float64)
    pad = 1
    img_pad = manual_pad_reflect(img.astype(np.float64), pad)
    tinggi, lebar = img.shape
    hasil = np.zeros((tinggi, lebar), dtype=np.float64)

    # konvolusi manual
    for i in range(tinggi):
        for j in range(lebar):
            total = 0.0
            for ki in range(3):
                for kj in range(3):
                    total += kernel[ki, kj] * img_pad[i+ki, j+kj]
            hasil[i, j] = total

    return manual_clip(hasil, 0, 255).astype(np.uint8)


# ============================================================
# FUNGSI 4: IMWRITE MANUAL (BMP 8-bit) — tetap sama
# ============================================================
def imwrite_manual(path, img):
    tinggi, lebar = img.shape
    img = img.astype(np.uint8)

    padding = (4 - (lebar % 4)) % 4
    ukuran_data = (lebar + padding) * tinggi
    offset_data = 14 + 40 + (256 * 4)
    ukuran_file = offset_data + ukuran_data

    idx = np.arange(256, dtype=np.uint8)   # masih pakai np.arange, tapi ini hanya untuk palet
    palet = np.zeros((256, 4), dtype=np.uint8)
    palet[:, 0] = idx
    palet[:, 1] = idx
    palet[:, 2] = idx

    if padding > 0:
        data_pad = np.zeros((tinggi, lebar + padding), dtype=np.uint8)
        data_pad[:, :lebar] = img
        data_piksel = data_pad[::-1].tobytes()
    else:
        data_piksel = img[::-1].tobytes()

    with open(path, 'wb') as f:
        f.write(b'BM')
        f.write(struct.pack('<I', ukuran_file))
        f.write(struct.pack('<H', 0))
        f.write(struct.pack('<H', 0))
        f.write(struct.pack('<I', offset_data))
        f.write(struct.pack('<I', 40))
        f.write(struct.pack('<i', lebar))
        f.write(struct.pack('<i', tinggi))
        f.write(struct.pack('<H', 1))
        f.write(struct.pack('<H', 8))
        f.write(struct.pack('<I', 0))
        f.write(struct.pack('<I', ukuran_data))
        f.write(struct.pack('<i', 0))
        f.write(struct.pack('<i', 0))
        f.write(struct.pack('<I', 256))
        f.write(struct.pack('<I', 256))
        f.write(palet.tobytes())
        f.write(data_piksel)
    return True


# ============================================================
# PROSES UTAMA (Grayscale → Resize → CLAHE → Sharpening)
# ============================================================
print("Memulai proses preprocessing (Grayscale → Resize → CLAHE → Sharpening) ...")

for label in KATEGORI:
    folder_input = os.path.join(PATH_INPUT, label)
    folder_output = os.path.join(PATH_OUTPUT, label)

    if not os.path.exists(folder_input):
        print(f"⚠️ Melewati folder (tidak ditemukan): {folder_input}")
        continue

    os.makedirs(folder_output, exist_ok=True)

    jumlah_sukses = 0
    for nama_file in tqdm(os.listdir(folder_input), desc=label):
        if nama_file.lower().endswith(('.jpg', '.jpeg', '.png')):
            jalur_masuk = os.path.join(folder_input, nama_file)
            nama_dasar = os.path.splitext(nama_file)[0]
            jalur_keluar = os.path.join(folder_output, nama_dasar + ".bmp")

            # 1. Baca grayscale
            img = cv2.imread(jalur_masuk, cv2.IMREAD_GRAYSCALE)
            if img is None:
                continue

            # 2. Resize ke 256x256
            img_resized = resize_manual(img, (256, 256))

            # 3. CLAHE
            img_clahe = clahe_manual(img_resized, clip_limit=2.0, grid_size=(8, 8))

            # 4. Sharpening
            img_sharp = sharpening_manual(img_clahe)

            # 5. Simpan
            imwrite_manual(jalur_keluar, img_sharp)
            jumlah_sukses += 1

    print(f"✅ Selesai memproses {jumlah_sukses} gambar di folder: {label}")

print(f"\n🎉 Selesai! Hasil ada di:\n{PATH_OUTPUT}")

Memulai proses preprocessing (Grayscale → Resize → CLAHE → Sharpening) ...


Normal: 100%|██████████| 100/100 [00:46<00:00,  2.13it/s]


✅ Selesai memproses 100 gambar di folder: Normal


kidneyStone: 100%|██████████| 100/100 [00:57<00:00,  1.75it/s]

✅ Selesai memproses 100 gambar di folder: kidneyStone

🎉 Selesai! Hasil ada di:
D:/semester 4/PCD/praktikum/projek/test/Assets_Prepro3/
